In [0]:
catalog = dbutils.widgets.text("catalog","hop")
catalog = dbutils.widgets.get("catalog")

# Support ticket ingestion by reader and writer api

In [0]:
df = spark.read.format("CSV").option("header", "true").option("mergeSchema", "true").load(f"/Volumes/{catalog}/bronze/data/raw/support_tickets.csv")
# Add metadata columns
df.write.mode("overWrite").saveAsTable(f"{catalog}.raw.support_tickets")

In [0]:
# ai_classify se tickets ko SOP-01 ke 6 issue codes me classify karo -> hop.genai.ticket_classified
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.genai")

labels = '''{
  "KITCHEN_DELAY": "The food was not ready when the rider arrived (restaurant fault)",
  "WRONG_ITEM": "A wrong or missing item in the order (restaurant fault)",
  "RIDER_DELAY": "The rider was slow or took a wrong route (rider fault)",
  "DAMAGED_IN_TRANSIT": "The food was spilled or crushed on the way (rider fault)",
  "CUSTOMER_UNAVAILABLE": "The customer could not be reached or gave a wrong address (customer fault)",
  "OTHER": "Anything else - fault unknown"
}'''

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.genai.ticket_classified AS
WITH tickets AS (
  SELECT ticket_id, description,
         ROW_NUMBER() OVER (PARTITION BY ticket_id ORDER BY created_ts DESC) AS rn
  FROM {catalog}.raw.support_tickets
)
SELECT
  ticket_id,
  COALESCE(
    ai_classify(
      description,
      '{labels}',
      MAP('version', '2.1', 'instructions', 'Inputs are HungryHop food delivery support ticket descriptions. Pick the single best-fit issue code.')
    ):response[0].value::STRING,
    'OTHER'
  ) AS predicted_issue_code
FROM tickets
WHERE rn = 1
""")

display(spark.sql(f"SELECT * FROM {catalog}.genai.ticket_classified"))

# SOP Chunking -> `hop.genai.sop_chunks`

## Kya karta hai ye cell
`/Volumes/hop/bronze/data/raw/sop_docs/*.md` ki 8 SOP files padh kar ek chunking table banata hai — Gen AI / RAG retrieval ke liye.

## Table schema
| Column | Type | Meaning |
| --- | --- | --- |
| `chunk_id` | STRING | `md5(doc_name :: chunk_index)` — stable ID, re-run par same chunk ka ID same rehta hai |
| `doc_name` | STRING | Source markdown file ka naam (e.g. `03_Late_Delivery_Compensation_Instant.md`) |
| `heading` | STRING | `##` section ka heading text (bina `##` prefix) |
| `chunk_index` | INT | Document ke andar section ka 0-based index |
| `text` | STRING | Chunk ka poora text |

## Chunking logic
* Har document mein **har `## section` = ek chunk** (8 docs x sections = 36 chunks).
* `text` hamesha isi order mein banta hai:
  1. Document ka `#` title line
  2. `Applies to:` line
  3. `## <heading>` line
  4. Section ka body (agle `##` tak)
* Isse har chunk self-contained hai — chunk ke bina bhi pata chalta hai kaunsi SOP, kis class pe apply hoti hai, aur kis section ki baat ho rahi hai.

## Notes
* Files `binaryFile` format se read hoti hain (serverless pe `wholetext` reliable nahi hai — line-by-line rows de deta hai).
* Table property `delta.enableChangeDataFeed = true` — SOP docs update hote hi downstream pipelines (e.g. vector index sync) change feed se incremental pickup kar sakte hain.

In [0]:
# SOP markdown files -> hop.genai.sop_chunks (one chunk per ## section, CDF on)
from pyspark.sql import functions as F
import hashlib

docs = (spark.read.format("binaryFile")
        .load(f"/Volumes/{catalog}/bronze/data/raw/sop_docs/*.md")
        .select(F.element_at(F.split(F.col("path"), "/"), -1).alias("doc_name"),
                F.decode(F.col("content"), "UTF-8").alias("content"))
        .collect())

chunks = []
for doc in docs:
    lines = doc["content"].splitlines()
    title = next(l for l in lines if l.startswith("# "))
    applies = next(l for l in lines if l.startswith("Applies to:"))
    heads = [i for i, l in enumerate(lines) if l.startswith("## ")]
    for idx, start in enumerate(heads):
        end = heads[idx + 1] if idx + 1 < len(heads) else len(lines)
        body = "\n".join(lines[start + 1:end]).strip()
        chunks.append((
            hashlib.md5(f'{doc["doc_name"]}::{idx}'.encode()).hexdigest(),
            doc["doc_name"],
            lines[start][3:].strip(),
            idx,
            f"{title}\n{applies}\n\n{lines[start]}\n{body}"
        ))

spark.createDataFrame(chunks, "chunk_id STRING, doc_name STRING, heading STRING, chunk_index INT, text STRING") \
     .createOrReplaceTempView("sop_chunks_tmp")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.genai.sop_chunks
AS SELECT chunk_id, doc_name, heading, chunk_index, text
FROM sop_chunks_tmp
ORDER BY doc_name, chunk_index
""")

spark.sql(f"ALTER TABLE {catalog}.genai.sop_chunks SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

display(spark.table(f"{catalog}.genai.sop_chunks"))

In [0]:
# Vector Search: endpoint + Delta Sync index (managed embeddings) over hop.genai.sop_chunks
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import vectorsearch
import time

w = WorkspaceClient()
endpoint_name = f"{catalog}_vs_endpoint"
index_name = f"{catalog}.genai.sop_chunks_idx"

# 1) Vector Search endpoint (pehli baar: create + ONLINE hone tak wait, ~2-4 min)
try:
    w.vector_search_endpoints.get_endpoint(endpoint_name)
except Exception:
    w.vector_search_endpoints.create_endpoint(name=endpoint_name, endpoint_type=vectorsearch.EndpointType.STANDARD)
    print("endpoint creating:", endpoint_name)

for _ in range(40):
    state = w.vector_search_endpoints.get_endpoint(endpoint_name).as_dict()["endpoint_status"]["state"]
    if state == "ONLINE":
        print("endpoint ONLINE:", endpoint_name)
        break
    time.sleep(15)
else:
    raise TimeoutError("endpoint ONLINE nahi hua — thodi der baad re-run karo")

# 2) Delta Sync index ke liye source table par PK constraint
spark.sql(f"ALTER TABLE {catalog}.genai.sop_chunks ALTER COLUMN chunk_id SET NOT NULL")
try:
    spark.sql(f"ALTER TABLE {catalog}.genai.sop_chunks ADD CONSTRAINT sop_chunks_pk PRIMARY KEY (chunk_id)")
except Exception as e:
    print("PK (already exists?):", str(e)[:200])

# 3) Delta Sync index — PK chunk_id, embedding source column `text`, model databricks-gte-large-en
try:
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="chunk_id",
        index_type=vectorsearch.VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=vectorsearch.DeltaSyncVectorIndexSpecRequest(
            source_table=f"{catalog}.genai.sop_chunks",
            embedding_source_columns=[vectorsearch.EmbeddingSourceColumn(
                name="text", embedding_model_endpoint_name="databricks-gte-large-en")],
            pipeline_type=vectorsearch.PipelineType.TRIGGERED,
        ),
    )
    print("index creating:", index_name)
except Exception as e:
    print("index create (already exists?):", str(e)[:200])

# 4) index READY hone tak wait (initial sync: 36 chunks embed ho rahe hain)
for _ in range(40):
    st = w.vector_search_indexes.get_index(index_name).as_dict().get("status") or {}
    if st.get("ready"):
        print("index READY:", st)
        break
    time.sleep(15)
else:
    print("Now, Index is making — get_index se status dekho")

In [0]:
# R11: Hybrid RAG (ANN+BM25) + LLM rerank + grounded answer over the SOP binder
import re, json, time, requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
CHAT_URL = f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1/chat/completions"
RERANK_MODEL = "system.ai.meta-llama-3-3-70b-instruct"  # gpt-oss-120b reasoning model hai — content blank deta tha
ANSWER_MODEL = "system.ai.meta-llama-3-3-70b-instruct"
JUDGE_MODEL = "system.ai.qwen3-next-80b-a3b-instruct"
NOT_IN_SOPS = "not in the SOPs"

def _chat(model, system, user, max_tokens=1024):
    body = {"model": model, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "max_tokens": max_tokens, "temperature": 0}
    r = None
    for delay in (0, 5, 15, 30, 60):  # 429/5xx pe exponential backoff retry
        if delay:
            time.sleep(delay)
        r = requests.post(CHAT_URL, headers={**w.config.authenticate(), "Content-Type": "application/json"},
                          json=body, timeout=180)
        if r.status_code == 200:
            return r.json()["choices"][0]["message"]["content"]
        if r.status_code not in (429, 500, 502, 503, 504):
            r.raise_for_status()
    r.raise_for_status()

def bare_chunk_id(doc_name, heading):
    # <file name>#<anchor> — heading lower case, non-alnum runs -> ek hyphen, ends pe hyphen nahi
    return f"{doc_name}#{re.sub(r'[^a-z0-9]+', '-', heading.lower()).strip('-')}"

def _hybrid_candidates(q, k=10):
    res = w.vector_search_indexes.query_index(
        index_name=f"{catalog}.genai.sop_chunks_idx",
        columns=["doc_name", "heading", "text"], query_text=q, query_type="HYBRID", num_results=k)
    return [{"bare_id": bare_chunk_id(r[0], r[1]), "text": r[2]} for r in res.result.data_array]

def _rerank(q, cands, k=5):
    listing = "\n".join(f"{i+1}. {c['bare_id']}\n{c['text'][:400]}" for i, c in enumerate(cands))
    try:
        out = _chat(RERANK_MODEL,
            "You rerank SOP sections for a food-delivery support question. A question about Instant orders "
            "must use sections whose Applies-to line says Instant orders; a question about Standard orders "
            "must use Standard sections. Reply ONLY a JSON array of candidate numbers, most relevant first, "
            "at most 5. Reply [] if no candidate answers the question.",
            f"Question: {q}\n\nCandidates:\n{listing}", max_tokens=200)
        s = out[out.find('['):out.rfind(']') + 1]
        nums = json.loads(s) if s.strip() else None
        if nums is None:  # blank/unparseable reply -> hybrid order fallback
            return cands[:k]
        picked, seen = [], set()
        for n in nums:
            try:
                n = int(n)
            except (TypeError, ValueError):
                continue
            if 1 <= n <= len(cands) and n not in seen:
                seen.add(n)
                picked.append(cands[n - 1])
        return picked[:k]  # [] => binder me answer nahi (model ne explicitly kaha)
    except Exception:
        return cands[:k]  # rerank fail -> hybrid order pe fallback

def answer_sop_question(q):
    """Answer + cited_chunk_ids (bare ids, top 5, rank order); exactly 'not in the SOPs' when binder has no answer."""
    top = _rerank(q, _hybrid_candidates(q))
    if not top:
        return {"question": q, "answer": NOT_IN_SOPS, "cited_chunk_ids": []}
    context = "\n\n---\n\n".join(f"[{c['bare_id']}]\n{c['text']}" for c in top)
    ans = _chat(ANSWER_MODEL,
        "You answer HungryHop support questions using ONLY the given SOP sections. Do not add anything. "
        f"If the sections do not answer the question, reply with exactly: {NOT_IN_SOPS}",
        f"Question: {q}\n\nSOP sections:\n{context}").strip()
    if NOT_IN_SOPS in ans.lower():
        return {"question": q, "answer": NOT_IN_SOPS, "cited_chunk_ids": []}
    return {"question": q, "answer": ans, "cited_chunk_ids": [c["bare_id"] for c in top]}

# demo: Instant vs Standard vs out-of-binder
for demo_q in ["What base coupon does a late Instant order earn?",
               "What base coupon does a late Standard order earn?",
               "What is the capital of France?"]:
    r = answer_sop_question(demo_q)
    print(f"Q: {demo_q}")
    print(f"  cited: {r['cited_chunk_ids']}")
    print(f"  answer: {r['answer'][:150]}")
    print()

# Gold Answers -> `hop.genai.gold_answers`

## Kya karta hai ye cell
Gold set ke **G01–G19** + **U01, U02** (out-of-binder) questions ka R11 pipeline se answer nikaal kar ek table banata hai.

## Schema
| Column | Type | Meaning |
| --- | --- | --- |
| `question_id` | STRING | G01..G19 / U01 / U02 |
| `question` | STRING | Question text |
| `answer` | STRING | RAG answer; binder me answer nahi ho to exactly `not in the SOPs` |
| `cited_chunk_ids` | ARRAY&lt;STRING&gt; | Bare chunk ids, best first, 1–5 har row mein |

## Citation rules (jaise expected hai)
* **G04, G05** → pehla cited chunk `03_Late_Delivery_Compensation_Instant.md#...` ka (Instant SOP first)
* **G06, G07** → pehla cited chunk `04_Late_Delivery_Compensation_Standard.md#...` ka (Standard SOP first)
* **U01, U02** → answer exactly `not in the SOPs` (binder me ye topics nahi hain); citations top retrieval sections hote hain kyunki har row me 1–5 ids chahiye

## Special case G19 (multi-hop sentinel)
G19 ko sirf binder se nahi, **order ke structured facts** (`hop.gold.orders_enriched` se class, promised/actual minutes, minutes_past_promise, order value, tier + `ticket_classified` se issue code) **+ matching SOP sections** ke saath answer kiya jata hai — late tha ya nahi, aur compensation kya milega.

In [0]:
# R11: hop.genai.gold_answers — gold set G01–G19 + U01/U02 answers with citations
import json
from pyspark.sql import functions as F

gold = json.loads(spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog}/bronze/data/raw/gold_set.json")
    .select(F.decode(F.col("content"), "UTF-8").alias("c")).first()["c"])

extras = [{"id": "U01", "question": "Does HungryHop deliver alcohol?"},
          {"id": "U02", "question": "What is the monthly salary of a HungryHop rider?"}]

def _answer_cited(q):
    # normal RAG; 'not in the SOPs' ho to bhi har row me 1-5 citations -> top retrieval (best first) cite karo
    r = answer_sop_question(q)
    if not r["cited_chunk_ids"]:
        r["cited_chunk_ids"] = [c["bare_id"] for c in _hybrid_candidates(q, k=5)]
    return r

def _answer_g19():
    # multi-hop: order ke structured facts (tables) + matching SOP sections (binder)
    gq = next(g for g in gold["questions"] if g["id"] == "G19")
    oe = f"{catalog}.gold.orders_enriched"
    oe_cols = {c.name.lower() for c in spark.table(oe).schema}
    tier_col = next((c for c in ("customer_tier", "tier") if c in oe_cols), None)
    fact = spark.sql(f"""
        SELECT o.order_id, o.status, o.delivery_class, o.promised_minutes,
               o.minutes_past_promise, o.actual_minutes, o.order_value
               {", o." + tier_col + " AS tier" if tier_col else ", NULL AS tier"}
        FROM {oe} o WHERE o.order_id = 'ORD-50003'""").first()
    issue = spark.sql(f"""
        SELECT c.predicted_issue_code FROM {catalog}.raw.support_tickets t
        JOIN {catalog}.genai.ticket_classified c ON c.ticket_id = t.ticket_id
        WHERE t.order_id = 'ORD-50003' LIMIT 1""").first()
    if fact is None:
        return _answer_cited(gq["question"])
    facts = (f"Structured facts - order {fact['order_id']}: status={fact['status']}, "
             f"delivery_class={fact['delivery_class']}, promised_minutes={fact['promised_minutes']}, "
             f"minutes_past_promise={fact['minutes_past_promise']}, actual_minutes={fact['actual_minutes']}, "
             f"order_value=Rs {fact['order_value']}, customer_tier={fact['tier']}"
             + (f", ticket_issue_code={issue['predicted_issue_code']}" if issue else ""))
    rq = (f"When does a {fact['delivery_class']} order count as late, what base coupon does a late "
          f"{fact['delivery_class']} order earn, and what coupon multiplier applies by membership tier?")
    top = _rerank(rq, _hybrid_candidates(rq)) or _hybrid_candidates(rq, k=5)
    context = "\n\n---\n\n".join(f"[{c['bare_id']}]\n{c['text']}" for c in top)
    ans = _chat(ANSWER_MODEL,
        "You answer HungryHop support questions using ONLY the given SOP sections and the order's structured "
        "facts. Do not add anything. If they do not answer the question, reply with exactly: " + NOT_IN_SOPS,
        f"Question: {gq['question']}\n\n{facts}\n\nSOP sections:\n{context}").strip()
    return {"question": gq["question"],
            "answer": ans if NOT_IN_SOPS not in ans.lower() else NOT_IN_SOPS,
            "cited_chunk_ids": [c["bare_id"] for c in top[:5]]}

rows = []
for gq in gold["questions"] + extras:
    r = _answer_g19() if gq["id"] == "G19" else _answer_cited(gq["question"])
    rows.append((gq["id"], gq["question"], r["answer"], r["cited_chunk_ids"]))

spark.createDataFrame(rows, "question_id STRING, question STRING, answer STRING, cited_chunk_ids ARRAY<STRING>") \
     .createOrReplaceTempView("gold_answers_tmp")
spark.sql(f"CREATE OR REPLACE TABLE {catalog}.genai.gold_answers AS SELECT * FROM gold_answers_tmp")

display(spark.table(f"{catalog}.genai.gold_answers"))

In [0]:
# R11 eval: known-section questions pe hit-rate@5 + MRR, answers ka LLM judge (failures plainly reported)
import json
from pyspark.sql import functions as F

gold = json.loads(spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog}/bronze/data/raw/gold_set.json")
    .select(F.decode(F.col("content"), "UTF-8").alias("c")).first()["c"])

rows, judge_failures = [], []
for gq in gold["questions"]:
    if gq["id"] == "G19":  # multi-hop sentinel — hit-rate@5 / MRR denominator se excluded (gold set ka note)
        continue
    r = answer_sop_question(gq["question"])
    exp = gq["expected_chunks"]
    rank = next((i + 1 for i, cid in enumerate(r["cited_chunk_ids"]) if cid in exp), None)
    verdict, jstat = None, "ok"
    try:
        out = _chat(JUDGE_MODEL,
            "You are an impartial judge. Reply with exactly one word: correct, partial, or incorrect.",
            f"Question: {gq['question']}\nExpected facts: {'; '.join(gq['expected_facts'])}\nAnswer: {r['answer']}",
            max_tokens=20).strip().lower()
        verdict = out if out in ("correct", "partial", "incorrect") else "unparseable"
        if verdict == "unparseable":
            jstat = f"unparseable judge reply: {out[:60]}"
            judge_failures.append((gq["id"], jstat))
    except Exception as e:
        jstat, verdict = f"judge call failed: {str(e)[:100]}", "judge_failed"
        judge_failures.append((gq["id"], jstat))
    rows.append((gq["id"], gq["question"], exp[0], rank, r["cited_chunk_ids"], r["answer"], verdict, jstat))

df = spark.createDataFrame(rows, "id STRING, question STRING, expected STRING, rank INT, cited ARRAY<STRING>, answer STRING, verdict STRING, judge_status STRING")

n = len(rows)
hits = [r for r in rows if r[3] is not None]
judged = [r for r in rows if r[7] == "ok"]
score_map = {"correct": 1.0, "partial": 0.5, "incorrect": 0.0}
judge_score = (sum(score_map.get(r[6], 0.0) for r in judged) / len(judged)) if judged else None

print(f"questions: {n} | hit-rate@5: {len(hits)}/{n} = {len(hits)/n:.1%} | MRR: {sum(1/r[3] for r in hits)/n:.3f}")
print(f"judged: {len(judged)}/{n} | judge score (correct=1, partial=0.5, incorrect=0): {judge_score:.3f}" if judge_score is not None else "judged: 0/n — sab judge calls fail hue")
print(f"JUDGE FAILURES: {len(judge_failures)}")
for jf in judge_failures:
    print("  ", jf)
display(df.select("id", "expected", "rank", "cited", "verdict", "judge_status"))

In [0]:
# R11 eval: known-section questions pe hit-rate@5 + MRR, answers ka LLM judge (failures plainly reported)
import json, re
from pyspark.sql import functions as F

gold = json.loads(spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog}/bronze/data/raw/gold_set.json")
    .select(F.decode(F.col("content"), "UTF-8").alias("c")).first()["c"])

rows, judge_failures = [], []
for gq in gold["questions"]:
    if gq["id"] == "G19":  # multi-hop sentinel — hit-rate@5 / MRR denominator se excluded (gold set ka note)
        continue
    r = answer_sop_question(gq["question"])
    exp = gq["expected_chunks"]
    rank = next((i + 1 for i, cid in enumerate(r["cited_chunk_ids"]) if cid in exp), None)
    verdict, jstat = None, "ok"
    try:
        out = _chat(JUDGE_MODEL,
            "You are an impartial judge. First reason briefly (1-2 lines) about whether the answer states "
            "the expected facts; extra true information taken from the SOPs is fine and must not lower the "
            "verdict. Then on the final line output exactly: Verdict: correct (expected facts stated), "
            "Verdict: partial (expected facts only partly stated), or Verdict: incorrect (expected facts "
            "missing or contradicted).",
            f"Question: {gq['question']}\nExpected facts: {'; '.join(gq['expected_facts'])}\nAnswer: {r['answer']}",
            max_tokens=150)
        m = re.findall(r"verdict:\s*(correct|partial|incorrect)", out.lower())
        verdict = m[-1] if m else "unparseable"
        if verdict == "unparseable":
            jstat = f"unparseable judge reply: {out[:60]}"
            judge_failures.append((gq["id"], jstat))
    except Exception as e:
        jstat, verdict = f"judge call failed: {str(e)[:100]}", "judge_failed"
        judge_failures.append((gq["id"], jstat))
    rows.append((gq["id"], gq["question"], exp[0], rank, r["cited_chunk_ids"], r["answer"], verdict, jstat))

df = spark.createDataFrame(rows, "id STRING, question STRING, expected STRING, rank INT, cited ARRAY<STRING>, answer STRING, verdict STRING, judge_status STRING")

n = len(rows)
hits = [r for r in rows if r[3] is not None]
judged = [r for r in rows if r[7] == "ok"]
score_map = {"correct": 1.0, "partial": 0.5, "incorrect": 0.0}
judge_score = (sum(score_map.get(r[6], 0.0) for r in judged) / len(judged)) if judged else None

# structured eval result: hit_rate_at_5 (number), mrr (number),
# judges_status ('ran' ya 'failed: <error>'), judges (object: per-question scores, non-empty jab ran)
hit_rate_at_5 = len(hits) / n
mrr = (sum(1 / r[3] for r in hits) / n) if hits else 0.0
judges = {r[0]: score_map.get(r[6], 0.0) for r in rows if r[7] == "ok"}
judges_status = "ran" if judges else ("failed: " + (judge_failures[0][1] if judge_failures else "no judge output"))

eval_result = {"hit_rate_at_5": hit_rate_at_5, "mrr": mrr,
               "judges_status": judges_status, "judges": judges}

print(f"questions: {n} | hit-rate@5: {len(hits)}/{n} = {hit_rate_at_5:.1%} | MRR: {mrr:.3f} | judges: {judges_status}")
print(f"JUDGE FAILURES: {len(judge_failures)}")
for jf in judge_failures:
    print("  ", jf)
print(json.dumps(eval_result, indent=2))
display(df.select("id", "expected", "rank", "cited", "verdict", "judge_status"))